# Clustering in Scanpy

In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
# os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1]# [0.1, 0.2, 0.3] # This takes a while per res
batch_col = 'sample' 
var_genes = 2000
n_pcs = 20
run_genes_in_smpl_filter = True

### Load and merge data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_dir, scanpy_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = load_and_dwnsmpl_data(None, True, scanpy_dir + 'adata_qc_plate1.h5ad', scanpy_dir + 'adata_qc_plate2.h5ad')
print(adata.shape)
print(adata.X[:10, :10].todense())

### Additional filter based on number of genes expressed per sample

In [ ]:
# Plot reads per gene distribution
plot_gene_read_distribution(adata)
plot_stacked_figure(adata, sample_column="sample", color_column='leiden_0.1', barplot=False, recalculate_columns=True)

In [ ]:
# Retain genes with at least min_reads in at least min_samples in the dataset
# Failed on min_reads=10, min_samples=100
#if run_genes_in_smpl_filter is True:
#    for min_reads in [1, 5, 10]:
#        for min_samples in [25, 50, 75, 100]:
#            mask = filter_genes_by_read_count(adata, min_reads=min_reads, min_samples=min_samples, inplace=False, verbose=True)
#            filtered_genes = adata.var.index[mask].tolist()
#            print(f"Filtered genes when min reads={min_reads}, min_samples={min_samples}: {len(filtered_genes)} remain.\n")
min_reads = 10
min_samples = 100
filter_genes_by_read_count(adata, min_reads = min_reads, min_samples = min_samples, inplace = True, verbose = True)
print(f"Filtered genes when min reads = {min_reads}, min_samples = {min_samples}: {adata.shape[1]} remain.\n")

# Recalculate QC metrics after filtering genes
#mt_genes = [gene for gene in adata.var_names if gene.upper().startswith("MT-")]
#print(f"Number of mitochondrial genes: {len(mt_genes)}")
#sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)

In [ ]:
# Plot reads per gene distribution
plot_gene_read_distribution(adata)
plot_stacked_figure(adata, sample_column = "sample", color_column = 'leiden_0.1', barplot = False, recalculate_columns = True)

# Initial cell counts

In [ ]:
# Cell counts by sample
print(f"Number of samples: {adata.obs['sample'].nunique()}")
adata.obs['sample'].value_counts()

In [ ]:
# Cell counts by sublibrary
adata.obs['sublibrary'] = [x[1] for x in adata.obs.index.str.split('__s')] 
adata.obs['sublibrary'].value_counts()

In [ ]:
# Cell counts by plate
adata.obs['plate'].value_counts()

In [ ]:
# Save original counts to layer
adata.raw = adata.copy()  # Store as raw for DE analysis
adata.layers["counts"] = adata.X.copy()  # Save raw counts
adata.obs

### Run Scanpy analysis

In [ ]:
# Normalise
logger.info("Normalising ...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# Get highly variable genes
logger.info("Detecting variable genes ...")
sc.pp.highly_variable_genes(adata, n_top_genes = var_genes, flavor = "seurat_v3", batch_key = batch_col)

adata = adata[:,adata.var.highly_variable]
sc.pp.scale(adata, max_value = 10)
sc.pl.highly_variable_genes(adata)

hvg_genes = sorted(adata.var_names[adata.var['highly_variable']])
with open(scanpy_dir + "highly_variable_genes.txt", "w") as f:
    for gene in hvg_genes:
        f.write(f"{gene}\n")

In [ ]:
# PCA
logger.info("PCA ...")
sc.tl.pca(adata, n_comps = n_pcs, use_highly_variable = True)
sc.pl.pca_variance_ratio(adata, log = True, n_pcs = n_pcs, save='') # scanpy generates the filename automatically

In [ ]:
# Create UMAP and Clustering
logger.info("Clustering and UMAP ...")
sc.pp.neighbors(adata, n_neighbors = 10, n_pcs = n_pcs)
sc.tl.umap(adata)

for res in resolutions:
    sc.tl.leiden(adata, resolution = res, key_added = f'leiden_{res}')

In [ ]:
# Plot UMAP
logger.info("Plot UMAP ...")
fig = create_umap_visualisations(adata, resolutions, leiden_prefix = "leiden", clustering_algorithm = "Leiden")
plt.show() 

In [ ]:
# UMAPs by .obs column
obs_columns = ['sublibrary', 'plate']
plot_umap_grid(adata, obs_columns, grid_size = (2, 2), figsize = (10, 8), save_path = None)

### Run batch correction

In [ ]:
# Store the original PCA and UMAP coordinates
adata.obsm['X_pca_pre_harmony'] = adata.obsm['X_pca']
adata.obsm['X_umap_pre_harmony'] = adata.obsm['X_umap']

# Run Harmony
sce.pp.harmony_integrate(adata, 'plate')

# Set PCA for clustering
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
sc.pp.neighbors(adata, n_neighbors = 10, n_pcs = n_pcs)
for res in resolutions:
    sc.tl.leiden(adata, resolution = res, key_added = f'leiden_harmony_{res}')
sc.tl.umap(adata) # Overwites X_umap

# Extract PCs as a DataFrame - need this for eQTL analysis expression covariates
expression_pcs = pd.DataFrame(adata.obsm['X_pca'], index = adata.obs.index, 
                              columns = [f'PC{i+1}' for i in range(n_pcs)])
expression_pcs.to_csv(scanpy_dir + "expression_pcs.csv")

In [ ]:
# Plot PCA before Harmony
#sc.pl.pca_scatter(adata, color='plate', title='PCA Before Harmony')

# Temporarily set the Harmony PCA for plotting
#adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']

# Plot PCA after Harmony
#sc.pl.pca_scatter(adata, color='plate', title='PCA After Harmony')

# Restore the original PCA if needed
#adata.obsm['X_pca'] = adata.obsm['X_pca_pre_harmony']

In [ ]:
# Plot Harmony Clusters
# Need to fix funct to deal with harmony and Non resolutions
#fig = create_umap_visualisations(adata, 0.3, leiden_prefix="leiden_harmony", clustering_algorithm="Leiden")
#plt.show() 
sc.pl.umap(adata, color=['leiden_harmony_0.1'], ncols = 3) 
adata.obsm

In [ ]:

# Need to add sex, pcw info etc.
# obs_columns = ['sublibrary', 'plate']
plot_umap_grid(adata, obs_columns, grid_size = (2, 2), figsize = (10, 8), save_path = None)

### Percentage of cells per sample per cluster

In [ ]:
# Default
plot_stacked_figure(adata, sample_column = "sample", color_column = "leiden_0.1", barplot = True, recalculate_columns = True)

In [ ]:
# Harmony
plot_stacked_figure(adata, sample_column = "sample", color_column = "leiden_harmony_0.1", barplot = True, recalculate_columns = True)

In [ ]:
# Plot sample per cluster
# Function saves an excel file with the cell counts per sample per cluster 
# Extract sample and leiden cluster information from the AnnData object
fig = plot_and_save_cluster_percentages(
    adata = adata,
    output_dir = scanpy_dir,
    clustering_param = "leiden_harmony_0.1"
)


### Extract PCs as a DataFrame - need this for eQTL analysis expression covariates

In [ ]:

expression_pcs = pd.DataFrame(adata.obsm['X_pca'], index = adata.obs.index, 
                              columns = [f'PC{i+1}' for i in range(n_pcs)])
expression_pcs.to_csv(scanpy_dir + "expression_pcs.csv")

### Final counts

In [ ]:
adata.obs[['total_counts', 'n_counts', 'n_genes']]

### Save

In [ ]:
logger.info("Saving h5ad file ...")
adata.write(scanpy_dir + f'adata_clusters.h5ad')
logger.info("All done.")